# FIP 04 — DA × NE cross-correlation and coherence across animals

Do DA and NE co-vary across animals, and at what lag and on which timescales?

The notebook uses every curated session with a same-hemisphere DA/NE pair in `DA_NE_4channels` and `DANE_3channels_curated`, holding each animal to one hemisphere. Per session it computes the cross-correlation and coherence against circular-shift nulls. Sessions are averaged within animal and then across animals, and each animal keeps its asset as `cohort`. The tests are Wilcoxon signed-rank on animal means, with hierarchical bootstrap intervals. The supplementary figure follows the coupling across recording days and within the session.

NE is GCaMP in LC noradrenergic axons, imaged in PL. DA is dLight in NAc (`latNAcc` in the newer asset). `su.norm_xcorr(da, ne)` peaks at a positive lag when NE leads DA.

## 1. Imports & setup

`fip_utils` (loading, curation), `signal_utils`, `stats_utils`, `plot_utils`.

In [1]:
import gc
import os
import pickle
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats

# autoreload so editing fip_utils.py doesn't cost a kernel restart + a full session reload.
%load_ext autoreload
%autoreload 2
import fip_utils as fu
import signal_utils as su
import stats_utils as st
import plot_utils as pu

from plotstyle import apply_style, PALETTE, style_ax, save_fig

%matplotlib inline
apply_style()

In [2]:
# ── Grid and session inclusion ────────────────────────────────────────────────
FS = 20                    # Hz, common grid for both signals
TASK_PAD_S = 10.0          # s kept after the last go cue; the grid starts at the first go cue
MIN_DURATION_S = 600.0     # s, shorter matched pairs are skipped
MAX_NONFINITE = 0.05       # skip a pair when either trace has more than this fraction non-finite
MIN_SESSIONS_PER_ANIMAL = 3
HEMI_BY_SUBJECT = {}       # override the per-animal side, e.g. {"808054": "R"}

# ── Assets ────────────────────────────────────────────────────────────────────
# use_curation: True applies the JSON curation at load (DA_NE_4channels); False for an asset
# Rachel already curated when she built it (DANE_3channels_curated). See section 2.
ASSETS = {
    "DA_NE_4channels": {"plot_loc": "/root/capsule/data/DA_NE_4channels/", "use_curation": True},
    "DANE_3channels_curated": {"plot_loc": "/root/capsule/data/DANE_3channels_curated/",
                               "use_curation": False},
}
USE_ASSETS = ["DA_NE_4channels", "DANE_3channels_curated"]

# Region labels that count as DA and as NE, one per naming scheme; `side` is the hemisphere.
DA_REGION_RE = re.compile(r"^(?:NAc|latNAcc)\((?P<side>[LR])\)-(?:dLight|DA)$")
NE_REGION_RE = re.compile(r"^PL\((?P<side>[LR])\)-(?:Gcamp|LCAxonCa)$")

# ── Cross-correlation and coherence ───────────────────────────────────────────
MAXLAG = 3.0               # s
N_SHIFT = 200              # circular shifts per session; p-value floor 1/(N_SHIFT+1)
MIN_SHIFT_S = 60.0         # s, smallest circular shift -- well past the 2-8 s shared timescale
COH_NPERSEG = 1024         # 51.2 s segments -> 0.02 Hz resolution
CLUSTER_ALPHA = 0.05
BAND_HZ = (0.12, 0.45)     # fip_03's coherence cluster, fixed in advance here (see section 4)

# ── Summaries ─────────────────────────────────────────────────────────────────
WIN_S, WIN_STEP_S = 120.0, 60.0   # windowed zero-lag correlation across the session
WIN_MAX_S = 7200.0
MIN_ANIMALS_PER_WINDOW = 3
N_BOOT = 10000             # hierarchical bootstrap: animals, then sessions within animal
SEED = 0

# ── Example session (None = the session whose peak r sits closest to the median) ──
EXAMPLE_SESSION = None
EXAMPLE_WINDOW_S = 180.0   # s of trace in panel a
EXAMPLE_ZOOM_S = 20.0      # s of trace in panel b, centred in panel a's window

# ── Output ────────────────────────────────────────────────────────────────────
FIG_DIR = "/root/capsule/scratch/figures/fip"  # note: /scratch is not kept by a reproducible run
SAVE_FIG = True

DA_COLOR, NE_COLOR, GRAND_COLOR = PALETTE["pos"], PALETTE["neg"], PALETTE["all"]
NULL_COLOR, SIG_COLOR = PALETTE["not_sig"], PALETTE["sig"]
DA_LABEL, NE_LABEL = "DA (NAc dLight)", "NE (PL LC-axon GCaMP)"
COHORT_TAG = {"DA_NE_4channels": "4ch", "DANE_3channels_curated": "3ch"}   # panel h / i labels
LAG_XLABEL = "lag (s)   —   positive = NE leads DA"

# Per-animal identity (colour + marker) comes from pu.animal_styles.

rng = np.random.default_rng(SEED)
K_LAG = int(round(MAXLAG * FS))
LAG_IDX = np.arange(-K_LAG, K_LAG + 1)
LAGS = LAG_IDX / float(FS)

## 2. Load and pair DA with NE by hemisphere

Each asset in `USE_ASSETS` is loaded, reduced to matched pairs and cached separately. `DA_NE_4channels` is curated at load (`use_curation=True`; labels like `NAc(L)-dLight`). `DANE_3channels_curated` was curated when built (`use_curation=False`; labels like `latNAcc(L)-DA`). `medNAcc` is not paired. Each pair is cut to the task and interpolated onto one `FS`-Hz grid.

In [3]:
def matched_pairs(meta):
    '''Same-side (DA, NE) region labels in one session.

    Parameters
    ----------
    meta : pandas.DataFrame
        From ``fu.build_meta``; must carry ``region``.

    Returns
    -------
    dict
        ``{side: (da_region, ne_region)}`` for each side matched by both ``DA_REGION_RE`` and
        ``NE_REGION_RE``.
    '''
    da, ne = {}, {}
    for region in sorted(meta["region"].dropna().unique()):
        m = DA_REGION_RE.match(region)
        if m:
            da.setdefault(m.group("side"), region)
        m = NE_REGION_RE.match(region)
        if m:
            ne.setdefault(m.group("side"), region)
    return {side: (da[side], ne[side]) for side in sorted(set(da) & set(ne))}


def best_event(meta, df_fip, region, prefer_variant="dff"):
    '''The df_fip event carrying exactly this region label.

    ``fu.pick_example``'s rule -- prefer a dff variant, then the most finite samples -- with an
    exact label match, because a substring such as ``'dLight'`` would match both sides.
    '''
    cand = meta[meta["region"] == region]
    pref = cand[cand["variant"].str.contains(prefer_variant, case=False, regex=False)]
    pool = pref if not pref.empty else cand
    return max(pool["event"],
               key=lambda ev: int(np.isfinite(fu.get_trace(df_fip, ev)[1]).sum()))


def finite_trace(df_fip, event):
    # (t, y, fraction of samples dropped as non-finite)
    t, y = fu.get_trace(df_fip, event, data_col="data")
    ok = np.isfinite(t) & np.isfinite(y)
    return t[ok], y[ok], 1.0 - ok.mean() if len(ok) else 1.0


def pairs_from_nwb_list(nwb_list, cohort):
    '''Every same-side DA/NE pair in one asset, on a common task-period grid.

    Parameters
    ----------
    nwb_list : list
        Sessions from ``fu.load_curated_sessions``.
    cohort : str
        Asset name, stored on each record.

    Returns
    -------
    tuple
        ``(records, skipped)`` -- one dict per matched pair, and a Counter of skip reasons.
    '''
    records, skipped = [], Counter()
    for nwb in nwb_list:
        df_fip = nwb.df_fip
        df_trials = getattr(nwb, "df_trials", None)
        if df_trials is None or fu.ALIGN_COL not in df_trials.columns:
            skipped["no df_trials"] += 1
            continue
        go = df_trials[fu.ALIGN_COL].dropna().to_numpy(float)
        if len(go) < 2:
            skipped["no go cues"] += 1
            continue

        meta = fu.build_meta(df_fip)
        if "region" not in meta.columns:
            skipped["no region labels"] += 1
            continue
        pairs = matched_pairs(meta)
        if not pairs:
            skipped["no same-side DA/NE pair"] += 1
            continue

        for side, (da_region, ne_region) in pairs.items():
            da_event = best_event(meta, df_fip, da_region)
            ne_event = best_event(meta, df_fip, ne_region)
            t_da, da, bad_da = finite_trace(df_fip, da_event)
            t_ne, ne, bad_ne = finite_trace(df_fip, ne_event)
            if max(bad_da, bad_ne) > MAX_NONFINITE:
                skipped["too many non-finite samples"] += 1
                continue

            lo = max(t_da.min(), t_ne.min(), go.min())
            hi = min(t_da.max(), t_ne.max(), go.max() + TASK_PAD_S)
            if hi - lo < MIN_DURATION_S:
                skipped["shorter than MIN_DURATION_S"] += 1
                continue

            grid = np.arange(lo, hi, 1.0 / FS)
            records.append({
                "cohort": cohort,
                "session_id": nwb.session_id,
                "subject_id": nwb.session_id.split("_")[0],
                "hemi": side, "da_region": da_region, "ne_region": ne_region,
                "da_event": da_event, "ne_event": ne_event,
                "t0": float(grid[0]),
                "da": np.interp(grid, t_da, da).astype(np.float32),
                "ne": np.interp(grid, t_ne, ne).astype(np.float32),
                "go_cues": go[(go >= grid[0]) & (go <= grid[-1])],
            })
    return records, skipped

In [4]:
# Loading an NWB list costs GBs and minutes per asset. Each asset's cache keeps only what later
# sections read -- one float32 trace per signal per matched pair, the grid start, the go cues
# and the labels, roughly 1 MB per pair. Set REBUILD = True after changing anything in the
# "Grid and session inclusion" or "Assets" blocks above.
REBUILD = False

records = []
for cohort in USE_ASSETS:
    cfg = ASSETS[cohort]
    cache = "/root/capsule/scratch/fip_04_pairs_%s.pkl" % cohort
    if not REBUILD and os.path.exists(cache):
        recs = pickle.load(open(cache, "rb"))
        print("%-24s %4d matched pairs from cache (written %s)"
              % (cohort, len(recs), time.ctime(os.path.getmtime(cache))))
    else:
        nwb_list = fu.load_curated_sessions(plot_loc=cfg["plot_loc"],
                                            use_curation=cfg["use_curation"])
        recs, skipped = pairs_from_nwb_list(nwb_list, cohort)
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        pickle.dump(recs, open(cache, "wb"), protocol=4)
        print("%-24s %d sessions -> %d matched pairs cached -> %s"
              % (cohort, len(nwb_list), len(recs), cache))
        print("%-24s skipped: %s" % ("", dict(skipped)))
        # Nothing below reads nwb_list; releasing it before the next asset loads frees GBs.
        del nwb_list
        gc.collect()
    records.extend(recs)

print("%d matched pairs from %s" % (len(records), ", ".join(USE_ASSETS)))
print(pd.DataFrame([{"cohort": r["cohort"], "da": r["da_region"], "ne": r["ne_region"]}
                    for r in records]).value_counts().to_string())

plot_loc = /root/capsule/data/DA_NE_4channels/
loading 7 df_sess file(s)
no bg_coef files found at plot location, skipping
no rpe_slope.csv found at plot location, skipping
loading 808054_2025-09-02
loading 808054_2025-09-03
loading 808054_2025-09-04
loading 808054_2025-09-05
loading 808054_2025-09-08
loading 808054_2025-09-09
loading 808054_2025-09-10
loading 808054_2025-09-11
loading 808054_2025-09-12
loading 808054_2025-09-15
loading 808054_2025-09-16
loading 808054_2025-09-17
loading 808054_2025-09-18
loading 808054_2025-09-19
loading 808054_2025-09-22
loading 808054_2025-09-23
loading 808054_2025-09-24
loading 808054_2025-09-25
loading 808054_2025-09-26
loading 808054_2025-10-02
loading 808054_2025-10-07
loading 808054_2025-10-08
loading 808054_2025-10-10
loading 808054_2025-10-13
loading 808054_2025-10-14
loading 808054_2025-10-15
loading 808054_2025-10-16
loading 808054_2025-10-17
loading 808056_2025-09-12
loading 808056_2025-09-15
loading 808056_2025-09-16
loading 808056_2025-0

### One side per animal

Each animal contributes its side with more sessions (or `HEMI_BY_SUBJECT`). Animals with fewer than `MIN_SESSIONS_PER_ANIMAL` sessions on that side are dropped.

In [5]:
cohort_of = {r["subject_id"]: r["cohort"] for r in records}
assert len({(r["subject_id"], r["cohort"]) for r in records}) == len(cohort_of), \
    "an animal appears in more than one asset -- decide which copy to keep before pooling"

avail = pd.DataFrame([{"subject": r["subject_id"], "session": r["session_id"], "hemi": r["hemi"]}
                      for r in records])
by_side = (avail.pivot_table(index="subject", columns="hemi", values="session",
                             aggfunc="count", fill_value=0)
           .reindex(columns=["L", "R"], fill_value=0))
by_side["side"] = [HEMI_BY_SUBJECT.get(subj, row[["L", "R"]].astype(int).idxmax())
                   for subj, row in by_side.iterrows()]
by_side["n_used"] = [int(row[row["side"]]) for _, row in by_side.iterrows()]
by_side["kept"] = by_side["n_used"] >= MIN_SESSIONS_PER_ANIMAL
by_side.insert(0, "cohort", [cohort_of[subj] for subj in by_side.index])

SIDE = by_side.loc[by_side["kept"], "side"].to_dict()
sessions = [dict(r) for r in records if SIDE.get(r["subject_id"]) == r["hemi"]]
sessions.sort(key=lambda s: (s["subject_id"], s["session_id"]))
for s in sessions:
    s["da"] = su.zscore(s["da"].astype(float))
    s["ne"] = su.zscore(s["ne"].astype(float))
    s["t"] = s["t0"] + np.arange(len(s["da"])) / float(FS)

SUBJECTS = sorted(SIDE)
N_ANIMALS = len(SUBJECTS)
SESSION_SUBJECTS = [s["subject_id"] for s in sessions]
ANIMAL_STYLE = pu.animal_styles(SUBJECTS)   # hue and marker cycle independently (6 x 8)

print(by_side.to_string())
print("\n%d sessions from %d animals | sides: %s | cohorts: %s"
      % (len(sessions), N_ANIMALS, dict(Counter(SIDE.values())),
         dict(Counter(cohort_of[subj] for subj in SUBJECTS))))
print("median session length %.1f min"
      % (np.median([len(s["da"]) for s in sessions]) / FS / 60.0))

hemi                     cohort   L   R side  n_used   kept
subject                                                    
800886   DANE_3channels_curated   5   0    L       5   True
808054          DA_NE_4channels   0  28    R      28   True
808056          DA_NE_4channels   0  40    R      40   True
809491          DA_NE_4channels  29   0    L      29   True
816212          DA_NE_4channels  27   0    L      27   True
816214          DA_NE_4channels  20   0    L      20   True
818585   DANE_3channels_curated   8   0    L       8   True
818586   DANE_3channels_curated  23   0    L      23   True
820688   DANE_3channels_curated   1   0    L       1  False
820691   DANE_3channels_curated   1   0    L       1  False

180 sessions from 8 animals | sides: {'L': 6, 'R': 2} | cohorts: {'DANE_3channels_curated': 3, 'DA_NE_4channels': 5}
median session length 74.9 min


### Averaging over animals

Grand means are means of animal means.

In [6]:
def grand_mean(values):
    '''Mean and SEM over animals of per-animal means (session means within animal first).

    Returns ``(mean, sem, per_animal)``; ``per_animal`` is ``(n_animals, ...)`` in ``SUBJECTS`` order.
    '''
    _, per_animal = st.group_means(values, SESSION_SUBJECTS)
    m, sem, _ = st.mean_sem(per_animal)
    return m, sem, per_animal


## 3. Cross-correlation per session

`su.xcorr_shift_null`: NE circularly shifted against DA by at least `MIN_SHIFT_S`. Per session: peak r, its lag, zero-lag r, median null peak |r|, shift p.

In [7]:
t_start = time.time()
for s in sessions:
    # + lag = DA later, i.e. NE leads; null_cc is (N_SHIFT, n_lags)
    lags, cc, null_cc = su.xcorr_shift_null(s["da"], s["ne"], FS, MAXLAG, N_SHIFT, MIN_SHIFT_S, rng)

    k = int(np.argmax(np.abs(cc)))
    null_peaks = np.max(np.abs(null_cc), axis=1)
    s["xcorr"], s["xcorr_null"] = cc, null_cc
    s["peak_r"], s["peak_lag"] = float(cc[k]), float(lags[k])
    s["r0"] = float(cc[K_LAG])
    s["peak_null_median"] = float(np.median(null_peaks))
    s["peak_p"] = st.shift_p(abs(cc[k]), null_peaks)
print("cross-correlation: %d sessions in %.0f s" % (len(sessions), time.time() - t_start))

df_sess = pd.DataFrame([{
    "subject": s["subject_id"], "cohort": s["cohort"], "session": s["session_id"],
    "side": s["hemi"],
    "minutes": len(s["da"]) / FS / 60.0,
    "peak_r": s["peak_r"], "peak_lag_s": s["peak_lag"], "r0": s["r0"],
    "null_median": s["peak_null_median"], "p_shift": s["peak_p"]} for s in sessions])
print("sessions above their own null at p<0.05: %d / %d"
      % (int((df_sess["p_shift"] < 0.05).sum()), len(df_sess)))
print("sessions peaking at exactly lag 0: %d / %d"
      % (int((df_sess["peak_lag_s"] == 0).sum()), len(df_sess)))

cross-correlation: 180 sessions in 12 s
sessions above their own null at p<0.05: 178 / 180
sessions peaking at exactly lag 0: 51 / 180


## 4. Coherence per session

Welch coherence (`COH_NPERSEG`-sample segments) with the same shift null (`su.coherence_shift_null`); cluster-mass test (`st.cluster_test`). `BAND_HZ` is fixed from `fip_03`; its mean coherence is one number per session. Above ~2 Hz both signals are at their noise floor.

In [8]:
t_start = time.time()
for s in sessions:
    f_coh, s["coh"], s["coh_null"] = su.coherence_shift_null(s["da"], s["ne"], FS, COH_NPERSEG,
                                                            N_SHIFT, MIN_SHIFT_S, rng)
print("coherence: %d sessions x %d shifts in %.0f s"
      % (len(sessions), N_SHIFT, time.time() - t_start))

IN_BAND = (f_coh >= BAND_HZ[0]) & (f_coh <= BAND_HZ[1])
for s in sessions:
    band_null = s["coh_null"][:, IN_BAND].mean(axis=1)
    s["band_coh"] = float(s["coh"][IN_BAND].mean())
    s["band_null_median"] = float(np.median(band_null))
    s["band_p"] = st.shift_p(s["band_coh"], band_null)

df_sess["band_coh"] = [s["band_coh"] for s in sessions]
df_sess["band_null_median"] = [s["band_null_median"] for s in sessions]
df_sess["band_p"] = [s["band_p"] for s in sessions]
print("sessions with %.2f-%.2f Hz coherence above their own null at p<0.05: %d / %d"
      % (BAND_HZ[0], BAND_HZ[1], int((df_sess["band_p"] < 0.05).sum()), len(df_sess)))

coherence: 180 sessions x 200 shifts in 300 s
sessions with 0.12-0.45 Hz coherence above their own null at p<0.05: 180 / 180


## 5. Grand means and animal-level tests

Grand-mean null: each null draw averaged within animal, then across animals. Tests: Wilcoxon signed-rank on animal means. Bootstrap: `aind_hierarchical_bootstrap` (animals, then sessions; session-weighted mean).

In [9]:
# Cross-correlation
xc_mean, xc_sem, xc_animal = grand_mean([s["xcorr"] for s in sessions])
xc_null = st.group_means([s["xcorr_null"] for s in sessions], SESSION_SUBJECTS)[1].mean(axis=0)  # (N_SHIFT, n_lags)
xc_null_lo, xc_null_hi = np.percentile(xc_null, [2.5, 97.5], axis=0)
k_grand = int(np.argmax(np.abs(xc_mean)))
grand_null_peaks = np.max(np.abs(xc_null), axis=1)
grand_peak_p = st.shift_p(abs(xc_mean[k_grand]), grand_null_peaks)

# Coherence
coh_mean, coh_sem, coh_animal = grand_mean([s["coh"] for s in sessions])
coh_null = st.group_means([s["coh_null"] for s in sessions], SESSION_SUBJECTS)[1].mean(axis=0)
coh_clusters, coh_thr, coh_crit = st.cluster_test(coh_mean, coh_null, f_coh, alpha=CLUSTER_ALPHA)

print("grand-mean cross-correlation: peak r = %+.3f at %+.2f s (shift p = %.4f, floor %.4f)"
      % (xc_mean[k_grand], LAGS[k_grand], grand_peak_p, 1.0 / (1 + N_SHIFT)))
if coh_clusters:
    df_coh_clusters = pd.DataFrame(coh_clusters)
    df_coh_clusters["timescale_s"] = 1.0 / df_coh_clusters["peak_Hz"]
    print("grand-mean coherence clusters:")
    print(df_coh_clusters.to_string(index=False, float_format=lambda v: "%.3f" % v))
else:
    print("grand-mean coherence: no cluster survives (critical mass %.3f)" % coh_crit)

grand-mean cross-correlation: peak r = +0.288 at -0.05 s (shift p = 0.0050, floor 0.0050)
grand-mean coherence clusters:
 lo_Hz  hi_Hz  n_bins  peak_coh  peak_Hz   mass     p  timescale_s
 0.000  4.922     253     0.299    0.156 10.754 0.005        6.400
 4.961 10.000     259     0.523    8.789 10.321 0.005        0.114


In [10]:
df_animal = (df_sess.groupby("subject")
             .agg(cohort=("cohort", "first"), side=("side", "first"),
                  n_sessions=("session", "size"),
                  peak_r=("peak_r", "mean"), peak_lag_s=("peak_lag_s", "mean"),
                  r0=("r0", "mean"), null_median=("null_median", "mean"),
                  band_coh=("band_coh", "mean"), band_null_median=("band_null_median", "mean"),
                  frac_sessions_sig=("p_shift", lambda p: float(np.mean(p < 0.05))))
             .reindex(SUBJECTS))


def wilcoxon_row(name, d):
    m, p, n = st.wilcoxon_animals(d)
    return {"test": name, "n_animals": n, "mean_difference": m, "p": p}


# Hierarchical bootstrap (aind_hierarchical_bootstrap): resample animals, then sessions within
# each. Its estimate pools the resampled sessions, so it is the session-weighted mean.
boot = {c: st.hier_bootstrap(df_sess, c, ("subject",), n_boot=N_BOOT, seed=SEED)
        for c in ("peak_r", "peak_lag_s", "band_coh")}

df_tests = pd.DataFrame([
    wilcoxon_row("|peak r| > null median", df_animal["peak_r"].abs() - df_animal["null_median"]),
    wilcoxon_row("peak lag != 0", df_animal["peak_lag_s"]),
    wilcoxon_row("band coherence > null median",
                 df_animal["band_coh"] - df_animal["band_null_median"]),
])
df_tests["grand_mean"] = [df_animal["peak_r"].mean(), df_animal["peak_lag_s"].mean(),
                          df_animal["band_coh"].mean()]
df_tests["boot_ci95 (session-weighted)"] = ["[%.3f, %.3f]" % tuple(np.percentile(boot[c], [2.5, 97.5]))
                                           for c in ("peak_r", "peak_lag_s", "band_coh")]

print("per-animal means (session means within animal):")
print(df_animal.to_string(float_format=lambda v: "%.3f" % v))
print("\nacross-animal tests (Wilcoxon signed-rank on animal means; CI = hierarchical bootstrap, "
      "session-weighted):")
print(df_tests.to_string(index=False, float_format=lambda v: "%.4f" % v))

per-animal means (session means within animal):
                         cohort side  n_sessions  peak_r  peak_lag_s     r0  null_median  band_coh  band_null_median  frac_sessions_sig
subject                                                                                                                                
800886   DANE_3channels_curated    L           5   0.312       0.000  0.310        0.022     0.240             0.006              1.000
808054          DA_NE_4channels    R          28   0.242       0.057  0.241        0.024     0.188             0.006              0.929
808056          DA_NE_4channels    R          40   0.624       0.018  0.620        0.026     0.544             0.006              1.000
809491          DA_NE_4channels    L          29   0.336      -0.036  0.334        0.023     0.201             0.007              1.000
816212          DA_NE_4channels    L          27   0.002      -0.163 -0.011        0.020     0.052             0.006              1.000


/opt/conda/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


## 6. Figure

- **a, b** — example traces, 3 min and a 20 s zoom; ticks are go cues.
- **c** — example cross-correlation against its shift null.
- **d** — example coherence; shading marks significant clusters.
- **e** — peak lag per animal (points: sessions; tick: animal mean).
- **f, g** — grand-mean cross-correlation and coherence ± SEM over animals; grey lines are animals.
- **h** — peak r per animal; grey bar spans the animal's null medians.

Panels e, h and i order animals by mean peak r.

In [ ]:
if EXAMPLE_SESSION is None:
    med = df_sess["peak_r"].median()
    ex = sessions[int(np.argmin(np.abs(df_sess["peak_r"].to_numpy() - med)))]
else:
    ex = next(s for s in sessions if s["session_id"] == EXAMPLE_SESSION)
EX_TAG = "%s %s" % (ex["session_id"], ex["hemi"])

# Panel a window: centred in the session. Panel b: centred in panel a.
t_mid = 0.5 * (ex["t"][0] + ex["t"][-1])
win_a = (t_mid - EXAMPLE_WINDOW_S / 2, t_mid + EXAMPLE_WINDOW_S / 2)
win_b = (t_mid - EXAMPLE_ZOOM_S / 2, t_mid + EXAMPLE_ZOOM_S / 2)
ex_clusters, ex_thr, _ = st.cluster_test(ex["coh"], ex["coh_null"], f_coh, alpha=CLUSTER_ALPHA)
print("example session: %s | peak r %+.3f at %+.2f s | %d coherence cluster(s)"
      % (EX_TAG, ex["peak_r"], ex["peak_lag"], len(ex_clusters)))


def plot_traces(ax, s, win, shade=None):
    m = (s["t"] >= win[0]) & (s["t"] < win[1])
    ax.plot(s["t"][m], s["da"][m], color=DA_COLOR, lw=1.0, label=DA_LABEL)
    ax.plot(s["t"][m], s["ne"][m], color=NE_COLOR, lw=1.0, label=NE_LABEL)
    go = s["go_cues"][(s["go_cues"] >= win[0]) & (s["go_cues"] < win[1])]
    ax.plot(go, np.ones(len(go)), "|", color=PALETTE["neutral"], ms=7, mew=1.2, label="go cue",
            transform=ax.get_xaxis_transform(), clip_on=False)
    if shade is not None:
        ax.axvspan(shade[0], shade[1], color=NULL_COLOR, alpha=0.2, lw=0)
    ax.set_xlim(win)
    ax.set_xlabel("time from first go cue (s)")
    ax.set_ylabel("signal (z)")
    style_ax(ax)


def shade_clusters(ax, clusters):
    for i, c in enumerate(clusters):   # each cluster separately: never min-to-max of a mask
        ax.axvspan(c["lo_Hz"], c["hi_Hz"], color=SIG_COLOR, alpha=0.15, lw=0, zorder=0,
                   label="significant cluster" if i == 0 else None)


ANIMAL_ORDER = list(df_animal["peak_r"].sort_values().index)   # ascending mean peak r, every strip panel
# One label per animal: subject, side, sessions, cohort.
TICK_LABELS = ["%s %s n=%d %s" % (subj, df_animal.loc[subj, "side"], df_animal.loc[subj, "n_sessions"],
                                  COHORT_TAG[df_animal.loc[subj, "cohort"]]) for subj in ANIMAL_ORDER]


def animal_strip(ax, col, ylabel, null_col=None, ref=None):
    '''pu.strip_by_animal with this notebook's order, styles, labels and null bars.'''
    pu.strip_by_animal(ax, df_sess, col, ANIMAL_ORDER, ylabel, styles=ANIMAL_STYLE,
                       null_col=null_col, null_style="range", ref=ref, tick_labels=TICK_LABELS)

In [ ]:
fig = plt.figure(figsize=(16, 13))
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.3, top=0.93, bottom=0.06)

# a -- example traces
ax = fig.add_subplot(gs[0, :2])
plot_traces(ax, ex, win_a, shade=win_b)
ax.set_title("a   Example session %s — %.0f s" % (EX_TAG, EXAMPLE_WINDOW_S), loc="left")
ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.02), ncol=3, frameon=False)

# b -- zoom
ax = fig.add_subplot(gs[0, 2])
plot_traces(ax, ex, win_b)
ax.set_title("b   Zoom, %.0f s" % EXAMPLE_ZOOM_S, loc="left")

# c -- example cross-correlation
ax = fig.add_subplot(gs[1, 0])
lo_c, hi_c = np.percentile(ex["xcorr_null"], [2.5, 97.5], axis=0)
ax.fill_between(LAGS, lo_c, hi_c, color=NULL_COLOR, alpha=0.35, lw=0,
                label="circular-shift null (95%)")
ax.plot(LAGS, ex["xcorr"], color=GRAND_COLOR, lw=2, label="observed")
ax.plot(ex["peak_lag"], ex["peak_r"], "o", color=GRAND_COLOR, ms=8)
ax.axvline(0, color="k", lw=0.6, ls=":")
ax.set_xlabel(LAG_XLABEL)
ax.set_ylabel("normalised cross-correlation")
ax.set_title("c   Example, peak r = %+.2f at %+.2f s" % (ex["peak_r"], ex["peak_lag"]),
             loc="left")
ax.legend(loc="upper left", fontsize=8)
style_ax(ax)

# d -- example coherence
ax = fig.add_subplot(gs[1, 1])
shade_clusters(ax, ex_clusters)
ax.semilogx(f_coh[1:], ex["coh"][1:], color=GRAND_COLOR, lw=1.2, label="observed")
ax.semilogx(f_coh[1:], ex_thr[1:], color=NULL_COLOR, lw=1.5, ls="--", label="95% null")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("magnitude-squared coherence")
ax.set_title("d   Example coherence", loc="left")
ax.legend(loc="upper right", fontsize=8)
style_ax(ax)

# e -- peak lag per animal
ax = fig.add_subplot(gs[1, 2])
animal_strip(ax, "peak_lag_s", "peak lag (s), + = NE leads DA", ref=0.0)
ax.set_title("e   Peak lag per animal (Wilcoxon vs 0 p = %.3f)" % df_tests.loc[1, "p"],
             loc="left")

# f -- grand-mean cross-correlation
ax = fig.add_subplot(gs[2, 0])
ax.fill_between(LAGS, xc_null_lo, xc_null_hi, color=NULL_COLOR, alpha=0.35, lw=0,
                label="circular-shift null (95%)")
for c in xc_animal:
    ax.plot(LAGS, c, color=PALETTE["neutral"], lw=0.7, alpha=0.5)
pu.plot_mean_sem(ax, LAGS, xc_animal, GRAND_COLOR, "mean over animals", lw=2)
ax.axvline(0, color="k", lw=0.6, ls=":")
ax.set_xlabel(LAG_XLABEL)
ax.set_ylabel("normalised cross-correlation")
ax.set_title("f   Grand-mean cross-correlation (n = %d animals)" % N_ANIMALS, loc="left")
ax.legend(loc="upper right", fontsize=8)
style_ax(ax)

# g -- grand-mean coherence
ax = fig.add_subplot(gs[2, 1])
shade_clusters(ax, coh_clusters)
for c in coh_animal:
    ax.semilogx(f_coh[1:], c[1:], color=PALETTE["neutral"], lw=0.7, alpha=0.5)
pu.plot_mean_sem(ax, f_coh[1:], coh_animal[:, 1:], GRAND_COLOR, "mean over animals", lw=2)
ax.semilogx(f_coh[1:], coh_thr[1:], color=NULL_COLOR, lw=1.5, ls="--", label="95% null")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("magnitude-squared coherence")
ax.set_title("g   Grand-mean coherence (n = %d animals)" % N_ANIMALS, loc="left")
ax.legend(loc="upper right", fontsize=8)
style_ax(ax)

# h -- per-animal peak r
ax = fig.add_subplot(gs[2, 2])
animal_strip(ax, "peak_r", "peak r", null_col="null_median", ref=0.0)
p_r = df_tests.loc[0, "p"]
ax.set_title("h   Peak r per animal (Wilcoxon vs null p = %.3f)" % p_r, loc="left")

fig.suptitle("DA (NAc dLight) x NE (PL LC-axon GCaMP), same hemisphere — %d sessions, "
             "%d animals (%s)" % (len(sessions), N_ANIMALS, " + ".join(USE_ASSETS)), y=0.975)
save_fig(fig, "fip_04_fig1_da_ne_xcorr", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 7. Supplementary

- **i** — mean coherence in `BAND_HZ` per animal.
- **j** — peak r across recording days (day 0 = each animal's first session).
- **k** — zero-lag r in `WIN_S`-s windows across the session, where at least `MIN_ANIMALS_PER_WINDOW` animals contribute.

In [13]:
WIN_STARTS = np.arange(0.0, WIN_MAX_S - WIN_S + 1e-9, WIN_STEP_S)
n_win = int(round(WIN_S * FS))
for s in sessions:
    r_win = np.full(len(WIN_STARTS), np.nan)
    for i, w0 in enumerate(WIN_STARTS):
        a = int(round((w0 - s["t0"]) * FS))
        if a < 0 or a + n_win > len(s["da"]):
            continue
        r_win[i] = np.corrcoef(s["da"][a:a + n_win], s["ne"][a:a + n_win])[0, 1]
    s["r_win"] = r_win

win_mean, win_sem, win_animal = grand_mean([s["r_win"] for s in sessions])
win_n = np.sum(np.isfinite(win_animal), axis=0)
win_ok = win_n >= MIN_ANIMALS_PER_WINDOW
win_centres_min = (WIN_STARTS + WIN_S / 2.0) / 60.0

# Days since each animal's first session, from the date in session_id ('808054_2025-09-02').
dates = pd.to_datetime(df_sess["session"].str.split("_").str[1], errors="coerce")
df_sess["day"] = (dates - dates.groupby(df_sess["subject"]).transform("min")).dt.days
day_rho = pd.Series({subj: stats.spearmanr(g["day"], g["peak_r"]).statistic
                     if len(g) >= 3 else np.nan
                     for subj, g in df_sess.groupby("subject")}).reindex(SUBJECTS)
w_day = stats.wilcoxon(day_rho.dropna())
print("Spearman rho(day, peak r) within animal: %s"
      % ", ".join("%s %+.2f" % (k, v) for k, v in day_rho.items()))
print("across animals vs 0: Wilcoxon W=%.1f, p=%.4f" % (w_day.statistic, w_day.pvalue))

Spearman rho(day, peak r) within animal: 800886 +0.30, 808054 -0.21, 808056 -0.68, 809491 -0.36, 816212 -0.44, 816214 +0.06, 818585 -0.55, 818586 -0.17
across animals vs 0: Wilcoxon W=5.0, p=0.0781


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

ax = axes[0]
animal_strip(ax, "band_coh", "mean coherence, %.2f-%.2f Hz" % BAND_HZ,
             null_col="band_null_median")
ax.set_title("i   Band coherence per animal (Wilcoxon vs null p = %.3f)"
             % df_tests.loc[2, "p"], loc="left")

ax = axes[1]
for subj in SUBJECTS:
    color, marker = ANIMAL_STYLE[subj]
    sub = df_sess[df_sess["subject"] == subj].sort_values("day")
    ax.plot(sub["day"], sub["peak_r"], color=color, marker=marker, ms=5, lw=1.2,
            label="%s  rho=%+.2f" % (subj, day_rho[subj]))
ax.axhline(df_sess["null_median"].mean(), color=NULL_COLOR, lw=1.2, ls="--")
ax.set_xlabel("days since animal's first session")
ax.set_ylabel("peak r")
ax.set_title("j   Peak r across days (Wilcoxon rho vs 0 p = %.3f)" % w_day.pvalue, loc="left")
ax.legend(fontsize=7, ncol=2)
style_ax(ax)

ax = axes[2]
for c in win_animal:
    ax.plot(win_centres_min, np.where(win_ok, c, np.nan), color=PALETTE["neutral"], lw=0.7,
            alpha=0.5)
pu.plot_mean_sem(ax, win_centres_min[win_ok], win_animal[:, win_ok], GRAND_COLOR,
                 "mean over animals", lw=2)
ax.axhline(0, color="k", lw=0.6, ls=":")
ax.set_xlabel("time from first go cue (min, window centre)")
ax.set_ylabel("zero-lag r, %.0f s windows" % WIN_S)
ax.set_title("k   Coupling within the session", loc="left")
ax.legend(fontsize=8)
style_ax(ax)

fig.tight_layout()
save_fig(fig, "fip_04_fig2_da_ne_supplementary", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()